### Setup

In [2]:
import time
import requests
import pandas as pd
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()
api_key = os.environ["ALPHA_VANTAGE_API_KEY"]

### Constants

In [39]:
base_url = "https://www.alphavantage.co/query"
col_map = {"1. open": "open", "2. high": "high", "3. low": "low", "4. close": "close", "5. volume": "volume"}

### Extraction Function

In [40]:
def extract_daily_prices(symbol: str, api_key: str, outputsize: str = "compact") -> pd.DataFrame:
    params = {"apikey": api_key, "function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": outputsize}
    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()

    if "Time Series (Daily)" not in payload:
        raise ValueError(f"Unexpected answer for {symbol}: {payload}")

    return (
        pd.DataFrame(payload["Time Series (Daily)"]).T
        .rename(columns=col_map)
        .astype({"open": float, "high": float, "low": float, "close": float, "volume": int})
        .reset_index()
        .rename(columns={"index": "date"})
        .assign(
            symbol=payload["Meta Data"]["2. Symbol"],
            date=lambda d: pd.to_datetime(d["date"]),
        )
    )

In [41]:
def extract_many(symbols: list[str], api_key: str, sleep_seconds: int = 15) -> pd.DataFrame:
    frames = []
    for i, symbol in enumerate(symbols):
        try:
            frames.append(extract_daily_prices(symbol, api_key))
            print(f"[{i+1}/{len(symbols)}] {symbol} OK")
        except Exception as e:
            print(f"[{i+1}/{len(symbols)}] {symbol} FALHOU: {e}")
        if i < len(symbols) - 1:
            time.sleep(sleep_seconds)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

### Test With 5 Tickers

In [ ]:
df = extract_many(["IBM", "AAPL", "MSFT", "GOOGL", "AMZN"], api_key=api_key)

[1/5] IBM OK
[2/5] AAPL OK
[3/5] MSFT OK
[4/5] GOOGL OK
[5/5] AMZN OK


In [45]:
df

,date,open,high,low,close,volume,symbol
0,2026-05-15,218.200,220.910,217.615,219.30,6154450,IBM
1,2026-05-14,215.545,220.960,215.000,218.37,5906199,IBM
2,2026-05-13,218.270,218.310,212.340,214.64,8361214,IBM
3,2026-05-12,224.430,224.430,219.220,219.22,6045225,IBM
4,2026-05-11,228.660,230.230,222.550,223.55,6677186,IBM
...,...,...,...,...,...,...,...
495,2025-12-29,231.940,232.600,230.770,232.07,19797909,AMZN
496,2025-12-26,232.035,232.990,231.180,232.52,15994726,AMZN
497,2025-12-24,232.130,232.950,231.330,232.38,11420543,AMZN
498,2025-12-23,229.055,232.445,228.730,232.14,29230233,AMZN
